# 05. 질문에 답하는 에이전트 (RAG 질의응답)

**목적**: 사용자가 질문하면 에이전트가 **RAG 로 근거를 찾아 출처와 함께 답한다.**

| 부분 | 무엇을 검색하나 | 어떻게 답하나 |
|---|---|---|
| 1. 사회자 (`Moderator`) | 룰북 PDF 의 규칙 페이지 (`type=rule` 필터) | 검색한 발췌만 근거로 답하고 출처 페이지를 붙인다. 룰북에 없으면 모른다고 한다 |
| 2. 해설 에이전트 (`mafia_qa.GameQA`) | 룰북 + **끝난 게임의 기록**(발언, 투표, 밤 행동, 속마음) + **외부 자료(위키백과)** | `create_agent` 가 질문에 맞는 **도구를 스스로 골라** 검색한 뒤 근거를 붙여 답한다 |

03 노트북의 AI 8명은 같은 RAG 를 "행동"에 쓰고, 이 노트북의 에이전트는 "질문에 답하는 데" 쓴다.

In [1]:
# ===== 0. 작업 폴더 맞추기 =====
# 이 노트북은 08_Mafia_Agent 폴더에서 실행한다고 가정한다.
# (VS Code에서 상위 폴더를 열었을 때도 동작하도록 폴더를 찾아 이동한다.)
import os
import sys
from pathlib import Path

if not Path("config.py").exists():
    if Path("08_Mafia_Agent/config.py").exists():
        os.chdir("08_Mafia_Agent")
    else:
        raise FileNotFoundError("08_Mafia_Agent 폴더에서 노트북을 열어주세요.")

sys.path.insert(0, os.getcwd())

import config

print("===== 작업 폴더 =====")
print(config.BASE_DIR)
print("LLM:", config.LLM_MODEL, "/ 임베딩:", config.EMBED_MODEL)

===== 작업 폴더 =====
C:\Users\USER\Desktop\수업\딥러닝\My_Langchain\08_Mafia_Agent
LLM: qwen3:8b / 임베딩: bge-m3


In [2]:
# ===== 1. 준비 =====
import json

import rag
import mafia_agents as agents
import mafia_qa

vs = rag.load_db()
print("===== 룰북 벡터 DB 로드 완료 (Chunk %d개) =====" % vs._collection.count())

===== 룰북 벡터 DB 로드 완료 (Chunk 23개) =====


## 1. 사회자: 룰북 RAG 질의응답

질문 -> `rag.search_rules()` (메타데이터 필터 `type=rule`, 상위 3개) -> LLM 이 발췌만 근거로 3문장 이내 답변 -> 출처 페이지 표시.
마지막 질문처럼 룰북에 없는 내용은 지어내지 않고 모른다고 답하는지 확인한다.

In [3]:
# ===== 2. 사회자에게 규칙 질문 =====
moderator = agents.Moderator(vs)

questions = [
    "의사는 자기 자신을 보호할 수 있나요?",
    "투표에서 동점이 나오면 어떻게 되나요?",
    "마피아는 언제 이기나요?",
    "이 게임에서 우승하면 상금이 있나요?",   # 룰북에 없는 질문
]
for q in questions:
    docs = rag.search_rules(vs, q, k=3)
    print("[질문]", q)
    print("   검색된 페이지:", ", ".join("p.%d %s" % (d.metadata["page"] + 1, d.metadata["title"]) for d in docs))
    print("[사회자]", moderator.explain_rule(q))
    print()

[질문] 의사는 자기 자신을 보호할 수 있나요?
   검색된 페이지: p.3 직업별 능력, p.4 진행 순서와 특수 규칙, p.4 진행 순서와 특수 규칙


[사회자] 네, 의사는 자기 자신을 보호할 수 있으며, 연속으로 보호할 수도 있습니다.
(룰북 p.3) (룰북 p.4)



[질문] 투표에서 동점이 나오면 어떻게 되나요?
   검색된 페이지: p.4 진행 순서와 특수 규칙, p.2 기본 규칙과 승리 조건, p.4 진행 순서와 특수 규칙


[사회자] 투표에서 동점이 나오면 동점자만 후보로 재투표를 1회 합니다. 그래도 동점이면 그날은 처형이 없습니다.
(룰북 p.2) (룰북 p.4)



[질문] 마피아는 언제 이기나요?
   검색된 페이지: p.2 기본 규칙과 승리 조건, p.4 진행 순서와 특수 규칙, p.3 직업별 능력


[사회자] 마피아팀이 이기는 조건은 살아있는 마피아 수가 살아있는 비마피아(경찰, 의사, 시민) 수 이상이 되는 경우입니다. 예를 들어 마피아 2 대 시민팀 2일 때입니다. 10일을 넘기면 무승부입니다.
(룰북 p.2) (룰북 p.3) (룰북 p.4)



[질문] 이 게임에서 우승하면 상금이 있나요?
   검색된 페이지: p.2 기본 규칙과 승리 조건, p.1 마피아 게임 룰북, p.1 마피아 게임 룰북


[사회자] 룰북에는 우승 시 상금에 대한 내용이 없습니다. 따라서 상금이 있는지 모릅니다.
(룰북 p.1) (룰북 p.2)



## 2. 해설 에이전트: 질문에 맞는 지식을 스스로 골라 검색

게임이 끝나면 그 판의 기록을 문서로 나눠 **두 번째 벡터 DB**(메모리 Chroma, bge-m3)에 넣는다.
에이전트(`create_agent`)는 질문을 보고 아래 도구 중 필요한 것을 골라 호출한다.

| 도구 | 검색 대상 | 쓰이는 질문 예 |
|---|---|---|
| `search_rulebook` | 룰북 RAG (`type=rule`) | "의사는 자기 자신을 보호해도 돼?" |
| `search_game_log` | 이번 판 기록 RAG (메타데이터 `kind`/`day`/`actor`/`target` 로 사람별 필터) | "마피아는 왜 박준수를 노렸어?" |
| `player_summary` | 코드가 계산한 사실 (역할, 득표, 밤 행동, 조사 결과) | "경찰은 누구를 조사했고 결과가 뭐였어?" |

- 게임 기록에는 밤 행동과 속마음 같은 **비밀이 들어 있어서, 게임이 끝난 뒤에만** 만든다 (`from_state()` 는 진행 중인 게임이면 거부한다).
- 에이전트가 도구를 쓰지 않고 답하려 하면, 코드가 두 저장소를 모두 검색해 근거를 준 뒤 다시 답하게 한다.
- 여기서는 `data/demo/` 에 저장된 실제 한 판(시민팀 2일차 승리)에 대해 질문한다.

In [4]:
# ===== 3. 끝난 게임으로 해설 에이전트 만들기 =====
qa = mafia_qa.GameQA.from_file(vs, config.DATA_DIR / "demo" / "demo_game.json")

kinds = {}
for m in qa.game_vs.get()["metadatas"]:
    kinds[m["kind"]] = kinds.get(m["kind"], 0) + 1
print("===== 게임 기록 벡터 DB: 문서 %d개 =====" % sum(kinds.values()))
print("종류별:", kinds)   # role / speech / vote / public / night / reason / thought

===== 게임 기록 벡터 DB: 문서 68개 =====
종류별: {'role': 8, 'public': 4, 'speech': 12, 'vote': 12, 'night': 2, 'reason': 18, 'thought': 12}


In [5]:
# ===== 4. 질문하기 (에이전트가 고른 도구가 함께 출력된다) =====
for q in [
    "경찰은 첫날 밤에 누구를 조사했고 결과가 뭐였어?",
    "마피아는 왜 첫날 밤에 박준수를 노렸어?",
    "정도현은 왜 1일차에 강태호에게 투표했어?",
    "의사는 자기 자신을 보호해도 돼?",
    "이번 판에서 오지환은 피자를 좋아했어?",   # 기록에 없는 질문
]:
    qa.ask(q)
    print()

[질문] 경찰은 첫날 밤에 누구를 조사했고 결과가 뭐였어?
    ┌─ 에이전트가 고른 도구
    │ player_summary(player='경찰')
    └─ 2.6초
[답변] 경찰 박준수는 1일차 밤에 강태호를 조사했고, 결과는 마피아였습니다.  

근거: player_summary(경찰)



[질문] 마피아는 왜 첫날 밤에 박준수를 노렸어?
    ┌─ 에이전트가 고른 도구
    │ search_game_log(query='1일차 밤', player='마피아')
    └─ 3.0초
[답변] 마피아는 첫날 밤에 박준수를 노린 이유는 그가 경찰이라는 직업을 가지고 있었기 때문입니다. 당시 마피아는 박준수를 경찰로 판단하고, 그가 시민팀에 위협이 될 수 있는 인물이라고 생각했기 때문입니다.

근거: 1일차 밤 처치 선택 이유 (게임 후 공개)



[질문] 정도현은 왜 1일차에 강태호에게 투표했어?
    ┌─ 에이전트가 고른 도구
    │ search_game_log(query='1일차 투표', player='정도현')
    └─ 3.6초
[답변] 정도현은 1일차에 강태호에게 투표한 이유는 강태호가 밤에 움직였다는 소문이 나왔고, 마피아일 가능성이 높아 보였기 때문입니다. 당시 강태호가 밤에 움직였다는 소문이 퍼졌고, 이에 따라 의심의 대상이 되었기 때문입니다.

근거: 1일차 투표 이유 (게임 후 공개)



[질문] 의사는 자기 자신을 보호해도 돼?
    ┌─ 에이전트가 고른 도구
    │ search_rulebook(question='의사는 자기 자신을 보호해도 돼?')
    └─ 3.0초
[답변] 의사는 자기 자신을 보호할 수 있습니다. 룰북 p.3에 따르면, 의사의 능력은 "매일 밤 살아있는 한 명을 보호합니다. 자기 자신도 보호할 수 있고, 같은 사람을 연속으로 보호해도 됩니다."라고 명시되어 있습니다.

근거: 룰북 p.3



[질문] 이번 판에서 오지환은 피자를 좋아했어?
    ┌─ 에이전트가 고른 도구
    │ (코드) 룰북 + 게임 기록 + 외부 자료 모두 검색(question='이번 판에서 오지환은 피자를 좋아했어?')
    └─ 2.8초
[답변] 기록에 없습니다.  
근거: 룰북 p.1, 1일차 낮 발언, 2일차 낮 발언, 2일차 투표 이유



### 직접 질문하기

`MY_QUESTION` 을 바꿔서 실행한다. 03 노트북에서 방금 끝난 게임에 질문하려면 같은 커널에서
`qa_live = mafia_qa.GameQA.from_state(vs, state)` 로 만든 뒤 `qa_live.ask("...")` 를 호출한다.

In [6]:
# ===== 5. 직접 질문 =====
MY_QUESTION = "의사는 누구였고 누구를 보호했어?"
result = qa.ask(MY_QUESTION)

[질문] 의사는 누구였고 누구를 보호했어?
    ┌─ 에이전트가 고른 도구
    │ game_overview()
    └─ 2.6초
[답변] 의사는 정도현이었고, 1일차 밤에는 자신을, 2일차 밤에는 윤서진을 보호했습니다.  
근거: game_overview (의사 정도현이(가) 보호한 사람 모음, 1일차 밤 정도현(자기 자신), 2일차 밤 윤서진)


### 외부 자료(위키백과)로 답하기

`data/external/` 의 위키백과 「마피아 (게임)」 한국어·영어 문서(CC BY-SA 4.0)를 별도 컬렉션(`mafia_background`)에 넣었다.
게임 자체에 대한 일반 지식(유래, 변형, AI 연구)을 물으면 에이전트가 `search_background` 를 골라 위키백과를 근거로 답한다.
우리 게임의 규칙은 룰북이 기준이므로, 게임 AI 와 사회자는 외부 자료를 보지 않는다.

In [7]:
# ===== 6. 외부 자료 질문 (위키백과) =====
print("외부 자료 chunk 수:", qa.bg_vs._collection.count())
print()
for q in [
    "마피아 게임은 누가 언제 만들었어?",
    "늑대인간 게임은 마피아랑 무슨 관계야?",
    "AI가 마피아 게임을 하는 연구가 있어?",
]:
    qa.ask(q)
    print()

외부 자료 chunk 수: 74



[질문] 마피아 게임은 누가 언제 만들었어?
    ┌─ 에이전트가 고른 도구
    │ search_background(question='마피아 게임은 누가 언제 만들었어?')
    └─ 3.3초
[답변] 마피아 게임은 1986년 모스크바 대학교의 심리학부 교수 디미트리 다비도프(Дми́трий Давы́дов)가 창안하였다. 그는 1987년 봄에 처음으로 마피아 게임을 심리학 부서에서 진행했고, 이후 이 게임은 유럽과 미국으로 퍼져나갔다.  

근거: 위키백과(마피아 (게임) - 역사)



[질문] 늑대인간 게임은 마피아랑 무슨 관계야?
    ┌─ 에이전트가 고른 도구
    │ search_background(question='늑대인간 게임은 마피아랑 무슨 관계야?')
    └─ 4.0초
[답변] 늑대인간 게임은 마피아 게임의 변형 중 하나로, 마피아 역할을 늑대인간으로 대체한 것입니다. 마피아 게임은 1986년 소련의 디마 다비도프가 창안했으며, 늑대인간이라는 주제를 추가한 것은 1997년 앤드루 플로트킨이었습니다. 이 두 게임은 모두 정보를 가진 소수와 정보를 가지지 못한 다수의 갈등을 모티브로 한 파티용 게임입니다.  

근거: 위키백과(마피아 (게임) - 역사, 개요)



[질문] AI가 마피아 게임을 하는 연구가 있어?
    ┌─ 에이전트가 고른 도구
    │ search_background(question='마피아 게임 AI 연구')
    └─ 2.8초
[답변] 마피아 게임은 AI 연구에서 주목받는 주제 중 하나입니다. 이 게임은 설득과 속임수 같은 인지적 요소가 포함되어 있어, 다중 에이전트 협업, 의도적 해석, 이성적 사고 이해 등의 AI 기술이 필요합니다.  

근거: 위키백과(마피아 (게임) - 인공지능 연구)



## 3. 정리

| | 일반 챗봇 | 사회자 | 해설 에이전트 |
|---|---|---|---|
| 지식 | 모델이 아는 것 | 룰북 RAG | 룰북 RAG + 게임 기록 RAG + 코드 계산 |
| 흐름 | 질문 -> 답 | 질문 -> 검색 -> 답 (고정) | 질문 -> **도구 선택** -> 검색 -> 답 |
| 근거 | 없음 | 룰북 페이지 | 룰북 페이지, 날짜별 기록 |
| 모를 때 | 지어낼 수 있음 | "룰북에 없다" | "기록에 없다" |

한계: 8B 모델이라 "이번 판 사실 + 규칙"처럼 두 가지를 한 번에 묻으면 한쪽만 답할 때가 있다. 질문을 나눠서 하면 정확하다.